# 02 — Prapemrosesan Data

Notebook ini memvalidasi snapshot kanonis dan menerapkan keputusan proposal. Teks asli dipertahankan apa adanya; hanya group_key yang dinormalisasi untuk pengelompokan. Tidak ada lexicon slang atau fitur diagnostik di input pelatihan.

Sumber yang diunduh hanya satu JSONL main ke /tmp, lalu SHA-256 diverifikasi. Akses sumber publik tidak memerlukan token Hugging Face. Tidak ada mirror Kaggle. Output /kaggle/working adalah output notebook privat.

| Unsur | Keputusan |
|---|---|
| Emoji, slang, URL, tanda baca, profanity, bahasa campuran | Pertahankan; tanpa pembersihan manual |
| Label toxicity dan polarization | Strict majority terpisah; seri di salah satu tugas mengecualikan baris |
| String kosong persis | Keluarkan hanya untuk pasangan (0, 0); whitespace-only dipertahankan |
| Noise/spam dan duplikat | Pertahankan |
| group_key | NFKC, casefold, trim, gabung whitespace; metadata split |
| Batas token | 512; truncation dan dynamic padding saat feature extraction |

Proposal mensyaratkan text berupa string dan menghentikan pemrosesan jika malformed. Pada snapshot ini, indeks 10295 memiliki text null dan label (0, 0). Notebook membuat laporan validasi dan manifest blocked, lalu selesai tanpa membuat sample.parquet. Penanganan null hanya berubah lewat amendemen metode.

In [ ]:
import hashlib, importlib.metadata, json, os, platform, tempfile, unicodedata, urllib.request
from pathlib import Path
import pandas as pd

REPO = "Exqrch/IndoDiscourse"
REVISION = "8af8c0f00b05633c05cc41fbd7650178ababc892"
FILENAME = "indotoxic2024_annotated_data_v2_final.jsonl"
EXPECTED_SHA = "94bb43546f5040a606a8d842694b7d490df043051377a6580e1e95ad2c7334e7"
EXPECTED_ROWS = 28448
COLUMNS = ["text_id", "annotators_id", "text", "initial_paragraph", "topic",
           "is_noise_or_spam_text", "related_to_election_2024", "toxicity", "polarized",
           "profanity_obscenity", "threat_incitement_to_violence", "insults",
           "identity_attack", "sexually_explicit"]
STRINGS = {"text_id", "text", "initial_paragraph", "topic"}
LISTS = set(COLUMNS) - STRINGS
NULLABLE = {"initial_paragraph", "topic", "is_noise_or_spam_text", "related_to_election_2024",
            "profanity_obscenity", "threat_incitement_to_violence", "insults",
            "identity_attack", "sexually_explicit"}
OUT = Path("/kaggle/working/preprocessing"); OUT.mkdir(parents=True, exist_ok=True)
REPORT, MANIFEST, SAMPLE = OUT / "validation_report.json", OUT / "preprocessing_manifest.json", OUT / "sample.parquet"
for stale in (REPORT, MANIFEST, SAMPLE, OUT / "sample.tmp.parquet"):
    stale.unlink(missing_ok=True)

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

def write_json(path, value):
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

def validate(row):
    problems = []
    if not isinstance(row, dict):
        return ["<row>"]
    if set(row) != set(COLUMNS):
        problems.append("<columns>")
    for col in COLUMNS:
        if col not in row:
            problems.append(col); continue
        value = row[col]
        if value is None and col in NULLABLE:
            continue
        if col in STRINGS and not isinstance(value, str):
            problems.append(col)
        if col in LISTS and not (isinstance(value, list) and all(isinstance(x, str) for x in value)):
            problems.append(col)
    ids = row.get("annotators_id")
    if isinstance(row.get("text_id"), str) and not row["text_id"].strip():
        problems.append("text_id")
    valid_ids = isinstance(ids, list) and bool(ids) and all(
        isinstance(value, str) and value.strip() for value in ids)
    if isinstance(ids, list) and not valid_ids:
        problems.append("annotators_id")
    for col in ("toxicity", "polarized"):
        votes = row.get(col)
        if isinstance(votes, list):
            valid_votes = bool(votes) and all(
                isinstance(value, str) and value in {"0", "1"} for value in votes)
            if not valid_votes:
                problems.append(col)
            if isinstance(ids, list) and len(votes) != len(ids):
                problems.append(col)
    return sorted(set(problems))

url = f"https://huggingface.co/datasets/{REPO}/resolve/{REVISION}/{FILENAME}"
CACHE_DIR = Path("/tmp/indodiscourse_preprocessing"); CACHE_DIR.mkdir(parents=True, exist_ok=True)
cache = CACHE_DIR / f"{REVISION}_{FILENAME}"
if not cache.exists():
    fd, name = tempfile.mkstemp(prefix="download-", suffix=".jsonl", dir=CACHE_DIR)
    os.close(fd); tmp = Path(name)
    try:
        with urllib.request.urlopen(url, timeout=120) as response, tmp.open("wb") as target:
            while chunk := response.read(1024 * 1024):
                target.write(chunk)
        if sha256(tmp) != EXPECTED_SHA:
            raise ValueError("SHA-256 berkas sumber tidak cocok.")
        os.replace(tmp, cache)
    finally:
        tmp.unlink(missing_ok=True)
raw_sha = sha256(cache)
if raw_sha != EXPECTED_SHA:
    raise ValueError("SHA-256 cache tidak cocok dengan revisi yang dipin.")
records, issues = [], []
with cache.open(encoding="utf-8") as stream:
    for index, line in enumerate(stream):
        try:
            row = json.loads(line)
        except json.JSONDecodeError:
            row = None; issues.append((index, "<json>"))
        issues.extend((index, col) for col in validate(row))
        records.append(row)
if len(records) != EXPECTED_ROWS:
    issues.append((None, "<row_count>"))
issues = sorted(set(issues), key=lambda item: (-1 if item[0] is None else item[0], item[1]))
issue_list = [{"row_index": index, "column": col} for index, col in issues]
versions = {"python": platform.python_version(), "pandas": pd.__version__, "pyarrow": version("pyarrow")}
report = {
    "status": "blocked" if issue_list else "valid",
    "source": {"repository": REPO, "configuration": "main", "revision": REVISION,
               "filename": FILENAME, "sha256": raw_sha, "expected_rows": EXPECTED_ROWS,
               "actual_rows": len(records)},
    "issue_count": len(issue_list), "issues": issue_list,
    "issue_count_by_column": {col: sum(item["column"] == col for item in issue_list)
                              for col in sorted({item["column"] for item in issue_list})},
    "versions": versions, "raw_cache": str(CACHE_DIR),
}
write_json(REPORT, report)

## Validasi dan hasil

Laporan masalah hanya memuat indeks baris dan nama kolom; nilai teks tidak ditulis. Jika validasi gagal, cabang blocked membuat manifest tanpa artefak sampel. Jika sumber kelak lolos setelah metode ditinjau, cabang siap menerapkan aturan label dan group key.

Saat ready, sampel berisi teks asli, text_id, row_id, dua label, dan group_key. Noise/spam serta duplikat dipertahankan. text_id disimpan sebagai metadata, bukan fitur model. Fitur diagnostik, ID anotator, dan demografi tidak dimasukkan.

**Implikasi untuk metodologi.** Status blocked berarti belum ada input untuk feature extraction. Jangan mengisi, membuang, atau mengubah null tanpa amendemen metode.

In [ ]:
if issue_list:
    write_json(MANIFEST, {"status": "blocked", "block_reason": "source_validation_failed",
        "dataset": report["source"], "validation_report": REPORT.name,
        "validation_report_sha256": sha256(REPORT), "issue_count": len(issue_list),
        "sample_artifact": None, "versions": versions})
    print("STATUS=blocked: validasi gagal; sample.parquet tidak dibuat.")
else:
    def majority(votes):
        ones = votes.count("1")
        return None if 2 * ones == len(votes) else int(2 * ones > len(votes))
    rows = []
    ledger = {"source_rows": len(records), "excluded_tie": 0, "excluded_empty_pair_00": 0, "kept": 0}
    for row_id, row in enumerate(records):
        toxicity, polarized = majority(row["toxicity"]), majority(row["polarized"])
        if toxicity is None or polarized is None:
            ledger["excluded_tie"] += 1; continue
        if row["text"] == "" and toxicity == 0 and polarized == 0:
            ledger["excluded_empty_pair_00"] += 1; continue
        normalized = unicodedata.normalize("NFKC", row["text"]).casefold()
        rows.append({"row_id": row_id, "text_id": row["text_id"], "text": row["text"], "toxicity_label": toxicity,
                     "polarized_label": polarized, "group_key": " ".join(normalized.split())})
    ledger["kept"] = len(rows)
    assert ledger["source_rows"] == ledger["excluded_tie"] + ledger["excluded_empty_pair_00"] + ledger["kept"]
    sample = pd.DataFrame(rows, columns=["row_id", "text_id", "text", "toxicity_label", "polarized_label", "group_key"])
    temporary = OUT / "sample.tmp.parquet"
    try:
        sample.to_parquet(temporary, index=False); os.replace(temporary, SAMPLE)
    except (ImportError, ValueError, OSError):
        temporary.unlink(missing_ok=True)
        report.update({"status": "blocked", "block_reason": "parquet_writer_unavailable_or_failed"})
        write_json(REPORT, report)
        write_json(MANIFEST, {"status": "blocked", "block_reason": report["block_reason"],
            "dataset": report["source"], "validation_report": REPORT.name,
            "validation_report_sha256": sha256(REPORT), "sample_artifact": None, "versions": versions})
        print("STATUS=blocked: Parquet tidak tersedia; sample tidak dibuat.")
    else:
        report.update({"status": "ready", "sample_rows": len(sample), "ledger": ledger,
                       "sample_columns": list(sample.columns),
                       "text_policy": "original preserved; no manual cleanup",
                       "diagnostic_features_in_sample": False})
        write_json(REPORT, report)
        artifact = {"filename": SAMPLE.name, "sha256": sha256(SAMPLE),
                    "row_count": len(sample), "columns": list(sample.columns)}
        write_json(MANIFEST, {"status": "ready", "dataset": report["source"],
            "validation_report": REPORT.name, "validation_report_sha256": sha256(REPORT),
            "sample_artifact": artifact, "ledger": ledger, "versions": versions})
        print("STATUS=ready: sampel dan manifest privat dibuat.")

## Handoff

Direktori output berisi laporan validasi dan preprocessing_manifest.json. Untuk snapshot saat ini, manifest blocked dan sample.parquet tidak dibuat. Feature extraction hanya membaca output privat jika manifest ready serta hash dan jumlah baris cocok. Jangan membuat Kaggle Dataset atau menyalin artefak turunan ke GitHub.